# 02 · 당뇨병 진행도 예측 (회귀)

**데이터**: sklearn `load_diabetes` — 442명의 환자, 10개 feature, 1년 뒤 당뇨병 진행도(continuous) 예측.

**분류 vs 회귀의 차이**:
- 분류: 양성/음성 같은 **카테고리**를 맞춤 → accuracy, AUROC
- 회귀: **숫자**를 맞춤 → MAE, MSE, R²

이 노트북에서는:
1. 기본 회귀 파이프라인
2. Cross-validation
3. 잔차 분석 (모델이 어디서 틀리는지)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes

np.random.seed(42)

data = load_diabetes(as_frame=True)
X = data.data
y = data.target

print(f'shape: X {X.shape}, y {y.shape}')
print(f'features: {list(X.columns)}')
print(f'target 범위: {y.min():.0f} ~ {y.max():.0f}')
X.describe()

> 💡 Feature들이 이미 centered & scaled 되어 있다 (sklearn이 미리 전처리한 버전).

In [ ]:
# Target 분포
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(y, bins=30)
axes[0].set_title('Target (disease progression) 분포')
axes[0].set_xlabel('progression score')

# 상관관계 히트맵
df_all = X.copy()
df_all['target'] = y
sns.heatmap(df_all.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1], cbar_kws={'shrink': 0.6})
axes[1].set_title('Feature ↔ Target correlation')
plt.tight_layout()
plt.show()

## 2. 여러 회귀 모델 비교

- **Linear Regression**: 가장 단순한 직선
- **Ridge**: L2 규제로 과적합 억제
- **Lasso**: L1 규제로 불필요한 feature를 0으로
- **Random Forest**: 비선형 + feature 상호작용

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
}

rows = []
for name, m in models.items():
    # 5-fold cross-validation (학습셋 안에서)
    cv_r2 = cross_val_score(m, X_train, y_train, cv=5, scoring='r2').mean()
    # 홀드아웃 테스트셋
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    rows.append({
        'model': name,
        'CV R² (train)': cv_r2,
        'Test MAE': mean_absolute_error(y_test, pred),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'Test R²': r2_score(y_test, pred),
    })

pd.DataFrame(rows).set_index('model').round(3)

## 3. 잔차 분석 — 모델이 어디서 틀리나

잔차(residual) = 실제값 - 예측값. 이 값이 랜덤해 보여야 모델이 '할 수 있는 일은 다 한' 상태.

In [ ]:
best = Ridge(alpha=1.0).fit(X_train, y_train)
pred = best.predict(X_test)
resid = y_test - pred

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(pred, y_test, alpha=0.6)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Predicted vs Actual')

axes[1].scatter(pred, resid, alpha=0.6)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual')
axes[1].set_title('Residual plot (패턴 있으면 bad sign)')
plt.tight_layout()
plt.show()

## 4. 계수 해석

Linear/Ridge/Lasso는 **각 feature의 계수**를 준다 → 설명 가능 (explainable).

In [ ]:
coefs = pd.Series(best.coef_, index=X.columns).sort_values()
coefs.plot(kind='barh', figsize=(6, 4))
plt.title('Ridge Regression 계수 (양수=진행도 증가에 기여)')
plt.axvline(0, color='k', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. 인사이트 & 질문

**관찰**:
- R²가 0.4~0.5 근처면 '잘 된다'가 아니라 '분산의 절반 정도만 설명' — 의료 예후 예측은 원래 어렵다.
- Random Forest가 Linear를 크게 못 이긴다면 → 이 데이터의 관계는 대부분 선형.

**AI agent에 물어볼 것**:
1. "Ridge의 L2 penalty가 계수를 어떻게 줄이는지 수식으로 보여줘"
2. "Cross-validation이 왜 단순 train/test split보다 안전한지 예시로 설명해줘"
3. "회귀에서 R²가 음수가 나올 수도 있어?"

### 다음 노트북
→ `03_rf_vs_xgboost.ipynb`: 트리 앙상블 두 강자 비교 + 하이퍼파라미터 튜닝